In [ ]:
import os
import sys
import seaborn as sns
import pandas as pd

repository_path = "/Users/user/Downloads/sift2_unbiased/"
functions_path  = os.path.join(repository_path, "code", "paper_figures", "functions")
sys.path.append(os.path.abspath(functions_path))

from update_index import update_index
from streamline_count import streamline_count
from extract_bundles import extract_bundles
from compute_groundtruth_bundles import compute_groundtruth_bundles
from sum_upper_triangle import sum_upper_triangle

from load_sift2_cross import load_sift2_cross
from load_sift2_diff import load_sift2_diff

from plot_total_errors_stacked_categories import plot_total_errors_stacked_categories

In [ ]:
# Number of categories to plot
N_CATS = 4  

# Build custom 4‐color palette :
GLOBAL_PALETTE = [
    "red",         
    "orange",  
    "steelblue",   
    "lightblue"    
]

order = ["Pipeline 1", "Pipeline 2", "Pipeline 3", "Pipeline 4"]

# Register it as Seaborn’s default
sns.set_palette(GLOBAL_PALETTE)

In [ ]:
# =============================
# PARAMETERS & PATHS
# =============================

# Choose phantom, tractography string, and ground-truth file identifiers
phantom = "disco3_central_lesion_snr20"   # options: disco3_crossing_bundles_snr20, disco3_single_bundle_snr20, disco3_central_lesion_snr20 
phantom_path = os.path.join(repository_path,"data","phantoms",phantom)
tcks_cross = "tracks"  
tcks_unbiased = "tracks_template"  

tcks_cross_filtered = tcks_cross + "_filtered"
tcks_unbiased_filtered = tcks_unbiased + "_filtered"

# SIFT2 Cross-sectional (formerly SIFT2 Absolute)
reg_basis_abs = "streamline"
reg_fn_abs = "gamma"
reg_strength_abs = 0.1

# SIFT2 Symmetric
reg_basis_sym = "streamline"
reg_strength_sym = 0.1

# SIFT2 Differential
reg_fn_diff = "dualinvbarr"
reg_basis_diff = "streamline"
reg_strength_diff = 0.1

In [ ]:
# Extract the true fibre count for each timepoint
fiber_count_tp1 = streamline_count(os.path.join(phantom_path,'orig/ground_truth/tp1/tracks_gt_tp1.tck'))

In [ ]:
# =============================
# LOAD GROUND TRUTH 
# =============================

# Load ground truth connectomes and compute their difference.
gt_tp1 = update_index(pd.read_csv(f'{phantom_path}/orig/ground_truth/tp1/gt_sift2_tp1.csv', header=None))
gt_tp2 = update_index(pd.read_csv(f'{phantom_path}/orig/ground_truth/tp2/gt_sift2_tp2.csv', header=None))
gt_tp_diff = (gt_tp2 - gt_tp1).fillna(0)
gt_tp_av = (gt_tp1 + gt_tp2) / 2

# Filtering two spurious streamlines in the original phantom (self assigned to node)
gt_tp1[gt_tp1 < 3] = 0
gt_tp2[gt_tp2 < 3] = 0

In [ ]:
# ============================= WITHOUT FALSE-POSITIVES =============================
# Load the pipelines WITHOUT false-positives.
tp1_cross_filtered, tp2_cross_filtered = load_sift2_cross(phantom_path, tcks_cross_filtered, reg_basis_abs, reg_fn_abs, reg_strength_abs, normalise=fiber_count_tp1)
tp1_diff_filtered, tp2_diff_filtered   = load_sift2_diff(phantom_path, tcks_unbiased_filtered, reg_basis_abs, reg_fn_abs, reg_strength_abs, reg_fn_diff, reg_basis_diff, reg_strength_diff, normalise=fiber_count_tp1)


# Compute absolute errors squared
error_cross_tp1_filtered =  abs(tp1_cross_filtered - gt_tp1)

# Compute differential squared errors 
tp_diff_diff_filtered = tp2_diff_filtered - tp1_diff_filtered
error_diff_filtered  = abs(tp_diff_diff_filtered - gt_tp_diff)


# ============================= WITH FALSE-POSITIVES =============================
# Load the pipelines WITH false-positives.
tp1_cross, tp2_cross = load_sift2_cross(phantom_path, tcks_cross, reg_basis_abs, reg_fn_abs, reg_strength_abs, normalise=fiber_count_tp1)
tp1_diff, tp2_diff   = load_sift2_diff(phantom_path, tcks_unbiased, reg_basis_abs, reg_fn_abs, reg_strength_abs, reg_fn_diff, reg_basis_diff, reg_strength_diff, normalise=fiber_count_tp1)

# Compute absolute squared errors 
error_cross_tp1 = abs(tp1_cross - gt_tp1)

# Compute differential squared errors 
tp_diff_diff = tp2_diff - tp1_diff
error_diff  = abs(tp_diff_diff - gt_tp_diff)

In [ ]:
# =============================
# BUNDLE DEFINITIONS VIA GROUND TRUTH
# =============================

# Determine ground truth bundles by comparing gt_tp1 and gt_tp2.
# This returns:
#  - true_bundles_with_effect: entries that change between timepoints.
#  - true_bundles_no_effect: nonzero entries that are unchanged.
#  - false_bundles: entries that are zero in both.
true_bundles_with_effect, true_bundles_no_effect, false_bundles = compute_groundtruth_bundles(gt_tp1, gt_tp2)

In [ ]:
# =============================
# EXTRACT ERROR BUNDLES FOR EACH PIPELINE
# =============================

# ============================= WITHOUT FALSE-POSITIVES =============================

# For SIFT2 Cross Tp1
error_cross_tp1_true_no_effect_filtered   = extract_bundles(error_cross_tp1_filtered, true_bundles_no_effect)
error_cross_tp1_true_with_effect_filtered = extract_bundles(error_cross_tp1_filtered, true_bundles_with_effect)
error_cross_tp1_false_filtered            = extract_bundles(error_cross_tp1_filtered, false_bundles)

# For SIFT2 Differential.
error_diff_true_no_effect_filtered   = extract_bundles(error_diff_filtered, true_bundles_no_effect)
error_diff_true_with_effect_filtered = extract_bundles(error_diff_filtered, true_bundles_with_effect)
error_diff_false_filtered            = extract_bundles(error_diff_filtered, false_bundles)


# ============================= WITH FALSE-POSITIVES =============================

# For SIFT2 Differential.
error_diff_true_no_effect   = extract_bundles(error_diff, true_bundles_no_effect)
error_diff_true_with_effect = extract_bundles(error_diff, true_bundles_with_effect)
error_diff_false            = extract_bundles(error_diff, false_bundles)

# For SIFT2 Cross Tp1
error_cross_tp1_true_no_effect   = extract_bundles(error_cross_tp1, true_bundles_no_effect)
error_cross_tp1_true_with_effect = extract_bundles(error_cross_tp1, true_bundles_with_effect)
error_cross_tp1_false            = extract_bundles(error_cross_tp1, false_bundles)




In [ ]:
# ============================= WITHOUT FALSE-POSITIVES =============================
errors_cross_tp1_filtered= [
    sum_upper_triangle(error_cross_tp1_true_with_effect_filtered) + sum_upper_triangle(error_cross_tp1_true_no_effect_filtered),
    sum_upper_triangle(error_cross_tp1_false_filtered),
]

# ============================= WITH FALSE-POSITIVES =============================
errors_cross_tp1 = [
    sum_upper_triangle(error_cross_tp1_true_with_effect) + sum_upper_triangle(error_cross_tp1_true_no_effect),
    sum_upper_triangle(error_cross_tp1_false),
]

labels = ["True (No Effect)", "False Positives"]
colors = ["#248CC8", "#D47263"]  # pastel green, blue, red
method_labels = ["without false positives", "with false positives"]
ylabel =  r"sum of errors ($\varepsilon_{i,\operatorname{j}}^{\mathrm{abs}}$)"


plot_total_errors_stacked_categories(
    [errors_cross_tp1_filtered, errors_cross_tp1],
    labels=labels,
    method_labels=method_labels,
    title=f"",
    custom_colors=colors,
    ylabel=ylabel
)

In [ ]:
# ============================= WITHOUT FALSE-POSITIVES =============================
errors_differential_filtered = [
    sum_upper_triangle(error_diff_true_with_effect_filtered),
    sum_upper_triangle(error_diff_true_no_effect_filtered),
    sum_upper_triangle(error_diff_false_filtered),
]

# ============================= WITH FALSE-POSITIVES =============================
errors_differential = [
    sum_upper_triangle(error_diff_true_with_effect),
    sum_upper_triangle(error_diff_true_no_effect),
    sum_upper_triangle(error_diff_false),
]
# === Labels and color palette ===

labels = ["True (With Effect)", "True (No Effect)", "False Positives"]
colors = ["#71AB48", "#248CC8", "#D47263"]  # pastel green, blue, red
method_labels = ["without false positives", "with false positives"]
ylabel =  r"sum of errors ($\varepsilon_{i,\operatorname{j}}^{\mathrm{diff}}$)"

# === Plot ===

plot_total_errors_stacked_categories(
    [errors_differential_filtered, errors_differential],
    labels=labels,
    method_labels=method_labels,
    title=f"",
    custom_colors=colors,
    ylabel=ylabel,
)